# Stage 2: Instance Leaf Segmentation via Marker-Controlled Watershed

Pipeline post-processing untuk memisahkan daun yang overlap dari output binary mask U-Net.

In [ ]:
import torch
from PIL import Image
from pathlib import Path
import glob

from stage2 import (
    WatershedConfig,
    Stage2Pipeline,
    Stage2Visualizer,
    Stage2Evaluator,
)

## 1. Configuration

In [ ]:
config = WatershedConfig(
    probability_threshold=0.5,
    min_distance=10,
    min_instance_area=100,
    compactness=0.0,
)
config

## 2. Load Stage 1 Checkpoint

In [ ]:
CHECKPOINT = "checkpoints/best_binary.pth"

pipeline = Stage2Pipeline.from_checkpoint(
    checkpoint_path=CHECKPOINT,
    config=config,
)
print(f"Model loaded from {CHECKPOINT}")
print(f"Device: {pipeline.predictor.device}")

## 3. Run Single Image

In [ ]:
image_paths = sorted(glob.glob("data/imgs/*.png"))
print(f"Found {len(image_paths)} images")

image = Image.open(image_paths[0]).convert("RGB")
result = pipeline.process(image)

print(f"Instance count: {result['instance_mask'].max()}")

## 4. Visualize Pipeline

In [ ]:
viz = Stage2Visualizer()
viz.show_pipeline(result)
viz.show_overlay(result)

## 5. Batch Processing

In [ ]:
sample_paths = image_paths[:20]
sample_images = [Image.open(p).convert("RGB") for p in sample_paths]

results = pipeline.process_batch(sample_images)

for i, (path, result) in enumerate(zip(sample_paths, results)):
    n = result["instance_mask"].max()
    print(f"{Path(path).name}: {n} instances")

## 6. Evaluate vs Ground Truth

In [ ]:
from stage2.preprocessing import Stage2Preprocessor

preprocessor = Stage2Preprocessor(target_size=config.target_size)
evaluator = Stage2Evaluator()

gt_masks = []
for p in sample_paths:
    mask_path = Path("data/masks") / Path(p).name
    if mask_path.exists():
        gt = preprocessor.preprocess_mask(Image.open(mask_path))
        gt_masks.append(gt)
    else:
        gt_masks.append(None)

valid = [(r, g) for r, g in zip(results, gt_masks) if g is not None]
if valid:
    metrics = evaluator.evaluate_batch(
        [v[0] for v in valid], [v[1] for v in valid]
    )
    print(f"Mean F1: {metrics['mean_f1']:.4f}")
    print(f"Mean IoU: {metrics['mean_iou']:.4f}")
    print(f"Mean Precision: {metrics['mean_precision']:.4f}")
    print(f"Mean Recall: {metrics['mean_recall']:.4f}")
    print(f"Mean Count Error: {metrics['mean_count_error']:.2f}")
else:
    print("No ground truth masks found for evaluation")

## 7. Parameter Experiments

In [ ]:
from stage2.prediction import Stage1Predictor

predictor = Stage1Predictor.from_checkpoint(CHECKPOINT)

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
min_distances = [5, 10, 15, 20]

print("Threshold | MinDist | Mean Instances")
print("-" * 40)
for t in thresholds:
    for md in min_distances:
        cfg = WatershedConfig(
            probability_threshold=t,
            min_distance=md,
        )
        p = Stage2Pipeline(predictor, config=cfg)
        batch_results = p.process_batch(sample_images[:5])
        avg_instances = sum(
            r["instance_mask"].max() for r in batch_results
        ) / len(batch_results)
        print(f"  {t:.1f}     |   {md:2d}    |   {avg_instances:.1f}")

## 8. Save Results

In [ ]:
output_dir = Path("predictions/stage2")
output_dir.mkdir(parents=True, exist_ok=True)

for i, (path, result) in enumerate(zip(sample_paths, results)):
    prefix = f"{i:04d}_{Path(path).stem}_"
    pipeline.save_results(result, str(output_dir), prefix=prefix)

print(f"Saved {len(results)} results to {output_dir}")